# Pre-Batching glTF Files for Optimized Load

The goal with this notebook will be to optimize the loading of our mep-fittings file to the scene. There are a handful of unique geometries in the scene and a multitude of transformation matrices. We should be able to efficiently store this information within our gltf file, while minimizing duplication.

In [87]:
import bpy
import pymeshlab
import numpy as np
import pygltflib

In [88]:
ms = pymeshlab.MeshSet()

In [89]:
# bpy.ops.import_scene.gltf(filepath="data/mep-fittings-hires.glb")

In [90]:
# print(bpy.data.objects)

In [91]:
# visited = set()
# visited_guid = set()
# total = 0
# total_guid = 0

# test_object = bpy.data.objects[100]
# np.array(test_object.matrix_world)

# for key, value in test_object.data.items():
#     print(f"{key}: {value} \n")

# "IFC_GUID" in test_object

# for obj in bpy.data.objects:
#     total += 1
    
#     if obj.type == 'MESH':
#         uid = obj.data.session_uid

#         if (uid not in visited):
#             visited.add(uid)
    

#     if ("IFC_GUID" in obj):
#         if (obj["IFC_GUID"] not in visited_guid):
#             total_guid += 1
#             visited_guid.add(obj["IFC_GUID"])

# print(len(visited))
# print(total)
# print(total_guid)

There are 567 unique meshes in the scene. 11617 of them have a GUID.

In [92]:
# def decimate_mesh(v_mat, f_mat, perc_red=0.0):
#     """
#     Reduce mesh complexity using quadric edge collapse and remap original indices.

#     Loads a mesh from a file, performs decimation to a target percentage, and 
#     reorders the original vertex and face matrices so that the decimated 
#     vertices occupy the leading indices.

#     Parameters
#     ----------
#     v_mat : ndarray
#         Original vertex array in numpy format.
#     f_mat : ndarray
#         Original face (trianlge) array in numpy format
#     perc_red : float, optional
#         Target reduction percentage for the decimation algorithm (0.0 to 1.0). 
#         Default is 0.0 (no reduction).

#     Returns
#     -------
#     v_org_rmp : ndarray
#         The original vertex matrix reordered to align with decimated indices.
#     f_org_rmp : ndarray
#         The original face matrix updated to reflect the reordered vertex indices.
#     v_dm : ndarray
#         The vertex matrix of the decimated mesh.
#     f_dm : ndarray
#         The face matrix of the decimated mesh.

#     Raises
#     ------
#     AssertionError
#         If the remapped original vertices do not align with the decimated 
#         vertex set.
#     """
    
#     ms = pymeshlab.MeshSet()
#     org_m = pymeshlab.Mesh(v_mat, f_mat)

#     ms.add_mesh(org_m)

#     ms.meshing_decimation_quadric_edge_collapse(optimalplacement=False, targetperc=perc_red) #preservetopology=True, qualitythr=0.5
#     ms.compute_normal_per_vertex()

#     m = ms.current_mesh()

#     v_dm = m.vertex_matrix()
#     f_dm = m.face_matrix()
#     n_dm = m.vertex_normal_matrix()

#     v_dict = { tuple(row): i for i, row in enumerate(v_dm) }
#     v_remapping = np.argsort(np.array([v_dict.get(tuple(row), np.inf) for row in v_mat ]))

#     v_org_rmp = v_mat[v_remapping]

#     v_inv_mapping = np.argsort(v_remapping)
#     f_org_rmp = v_inv_mapping[f_mat]

#     ms.clear()

#     return v_dm, f_dm, n_dm, v_org_rmp, f_org_rmp

In [93]:
# matrix = None
# ifc_guid = None
# v_matrix = None
# f_matrix = None

# visited = set()
# visited_guid = set()

# for obj in bpy.data.objects:
#     if "IFC_GUID" in obj:
        
#         if obj["IFC_GUID"] not in visited_guid:
            
#             matrix = obj.matrix_world
#             ifc_guid = obj["IFC_GUID"]

#             uid = obj.data.session_uid

#             if uid not in visited:
#                 visited.add(uid)
                
                

In [94]:
# filename = "mep-fittings_ifc-guid.glb"
# gltf = pygltflib.GLTF2().load(filename)

In [95]:
# count_meshes = 0
# visited_mesh = set()

# # mesh = gltf.meshes[100]
# for mesh in gltf.meshes:
#     count_meshes += 1
#     # print(mesh.name)
#     if mesh.name not in visited_mesh:
#         visited_mesh.add(mesh.name)
#     else:
#         print(mesh.name)

# # for primitive in mesh.primitives:
#     # print(primitive.attributes.POSITION)

# print(count_meshes)
# print(len(visited_mesh))

All the meshes appear to be unique in the scene. This will be where we acquire our information.

In [96]:
# test_mesh = gltf.meshes[10]

# for primitive in mesh.primitives:
#     print(primitive)

In [97]:
# triangles_binary_blob = gltf.binary_blob()

# triangles_accessor = gltf.accessors[gltf.meshes[0].primitives[0].indices]
# triangles_buffer_view = gltf.bufferViews[triangles_accessor.bufferView]
# triangles_test = np.frombuffer(
#     triangles_binary_blob[
#         triangles_buffer_view.byteOffset
#         + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
#         + triangles_buffer_view.byteLength
#     ],
#     dtype="uint8",
#     count=triangles_accessor.count,
# ).reshape((-1, 3))

# triangles_test

In [98]:
# gltf_export = pygltflib.GLTF2()
# gltf_export.scenes.append(pygltflib.Scene(nodes=[]))
# gltf_export.scene = 0

In [99]:
# main_binary_blob = bytearray()

In [100]:
# triangles_binary_blob = f_matrix.flatten().tobytes()
# points_binary_blob = v_matrix.tobytes()
# normals_binary_blob = n_matrix.tobytes()

# main_binary_blob = triangles_binary_blob + points_binary_blob + normals_binary_blob

## Plan of Attack

Looks like there is a lot of unnecessary information within our file. We will work to iterate over this and save only the necessary info. This includes -->

Nodes -->
1) IFC_GUID
2) Transformation Matrix
3) Mesh_ID

Mesh -->
1) POSITION
2) INDEX
3) Normal

Attributes and buffers should remain untouched.

We start by creating a blank glTF file.

In [101]:
# # Initialize empty GLTF container to store the newly created mesh objects.
# gltf_export = pygltflib.GLTF2()
# gltf_export.scenes.append(pygltflib.Scene(nodes=[]))
# gltf_export.scene = 0

Create a binary blob bytearray.

In [102]:
# # Empty array for appending binary blobs
# main_binary_blob = bytearray()

We will need a series of index counters to manage our gltf routing.

In [103]:
# bufferView_counter = 0
# accessor_counter = 0
# mesh_counter = 0
# node_counter = 0

Need a Set object to manage our meshes.

In [104]:
# visited = set()

Now, our main loop.

In [105]:
# for node in gltf.nodes:
#     mesh = node.mesh

#     for primitive in gltf.meshes[mesh].primitives:
#         print(primitive)

In [106]:
# for node in gltf.nodes:
#     guid = node.extras["IFC_GUID"]

#     node.extras = None
#     node.name = guid


#     print(node)

# print(node_counter)

In [107]:
# for mesh in gltf.meshes:
#     mesh.extras = None

In [108]:
# for mesh in gltf.meshes:
#     print(mesh)

In [109]:
# filename = "mep-fittings_ifc-guid.glb"
# gltf.save(filename)

Apparently there are some extras saved at the scene level as well.

In [110]:
# for scene in gltf.scenes:
#     scene.extras = None
#     scene.extras = {
#         "nInstances": 11617,
#         "vCount": 14701928,
#         "iCount": 23408661
#     }
#     print(scene)

In [111]:
# filename = "mep-fittings_ifc-guid.glb"
# gltf.save(filename)

Lets do the same for our hyperinstanced MEP straight pipe file.

In [112]:
# gltf_mep = pygltflib.GLTF2().load("../data/mep-hyperinstance-ifc-guid-color.glb")

In [113]:
# for scene in gltf_mep.scenes:
#     scene.extras = {
#         "nInstances": 10957,
#         "vCount": 64,
#         "iCount": 192
#     }
#     print(scene)

In [114]:
# filename = "mep-hyperinstance-ifc-guid-color-2.glb"
# gltf_mep.save(filename)

## Optimized LOD Generation

We have previously implemented a technique where the lowres geometry of the mesh was included within our original mesh. This process involved first decimating our mesh, then reordering our original mesh's vertices to match the order of our decimated one. This way, the decimated mesh is an exact subset of the remapped original, and we only need to save the original mesh's geometry data to the scene. That said, the indices of the 2 meshes will still be different, and will need their own standalone data object.

Let's think through how we might ideally achieve this.

Our LOD engine in the scene will use an accelerated spatial query to determine which objects are near and far. We have previously implemented a scene where the geometry of the LOD is entirely swapped once we move on past a zoom level. This uses the `batchedMesh.setGeometryIdAt()` method, and required us maintaining an array of geometries to be able to keep track of the objects that were currently in hi-res or low-res.

I'd like to explore making our low-res-meshes the child of our existing hi-res one, or vice versa. This way when we create our batchedMesh, we should be able to simply add and subtract indices when needing to swap geometries. For example, when building the scene, we account for the additional instances, but only keep track of the hi-res number of vertices in the scene, since these will be reused.

### Plan

Here is what I'd like to do. At the node level, we create a parent that contains the low-res geometry of our mesh. A child of this node contains the hi-res geometry. Since these contain the same attributes, they should link to the same buffer uri, only differing in their bytelength. The normals should follow the same structure. Only the indices will be different.

1) Loop over all the nodes in the original file.
2) If the mesh has been visited before, reuse the same one.
3) If we are dealing with a new mesh, create a low-res LOD of the same and structure the vertex array to match the original. Also append the index array.
4) Add this data to the buffer, creating bufferviews and attributes in the process.
5) On to the next one.

In [115]:
filename = "mep-fittings_ifc-guid.glb"
gltf = pygltflib.GLTF2().load(filename)

In [116]:
# Initialize empty GLTF container to store the newly created mesh objects.
gltf_export = pygltflib.GLTF2()
gltf_export.scenes.append(pygltflib.Scene(nodes=[]))
gltf_export.scene = 0

In [117]:
# Empty array for appending binary blobs
main_binary_blob = bytearray()

In [118]:
bufferView_counter = 0
accessor_counter = 0
mesh_counter = 0
node_counter = 0

In [119]:
visited = set()

In [120]:
binary_blob = gltf.binary_blob()

In [121]:
gltf_test = pygltflib.GLTF2()
gltf_test.scenes.append(pygltflib.Scene(nodes=[]))
gltf_test.scene = 0

In [122]:
gltf_test.nodes.extend([
    pygltflib.Node(
        mesh = 0,
        name = "parent",
        children = [
            pygltflib.Node(
                mesh = 1,
                name = "child"
            )
        ]
    )
])

In [123]:
gltf_test.nodes[0]

Node(extensions={}, extras={}, mesh=0, skin=None, rotation=None, translation=None, scale=None, children=[Node(extensions={}, extras={}, mesh=1, skin=None, rotation=None, translation=None, scale=None, children=[], matrix=None, camera=None, name='child')], matrix=None, camera=None, name='parent')

Let's work through this sequentially. At highest level, we loop over all nodes in the scene. For each node in our original file, we create a new node in our export file. The total number of nodes in each file will be the same. However, we will also need to create a child node for each node. This one will contain the low-res geometry of the mesh. This will need to also be saved as a new mesh that links to new attributes and bufferviews, but which ultimately links back to the same buffer.

Here is some pseudo code that creates a parent and child node within a new file.

In [124]:
# Pseudo code
## Nodes

'''
for node in gltf.nodes:
    gltf_export.nodes.extend([
        pygltflib.Node(
            mesh = mesh_counter,
            rotation = node.rotation,
            scale = node.scale,
            name = node.name,
            children = [
                pygltflib.Node(
                    mesh = mesh_counter + 1,
                    rotation = node.rotation,
                    scale = node.scale,
                    name = node.name
                )
            ]
        )
    ])

    mesh_counter += 2

'''

'\nfor node in gltf.nodes:\n    gltf_export.nodes.extend([\n        pygltflib.Node(\n            mesh = mesh_counter,\n            rotation = node.rotation,\n            scale = node.scale,\n            name = node.name,\n            children = [\n                pygltflib.Node(\n                    mesh = mesh_counter + 1,\n                    rotation = node.rotation,\n                    scale = node.scale,\n                    name = node.name\n                )\n            ]\n        )\n    ])\n\n    mesh_counter += 2\n\n'

Now, we need to keep track of the visited meshes since we do not want to duplicate the work. Let's add a dictionary that will keep track of the mesh IDs we have already seen.

In [125]:
# Pseudo code
## Visited

'''
visited = {}

for node in gltf.nodes:
    mesh = node.mesh

    if mesh not in visited:
        visited[mesh] = mesh_counter

        gltf_export.nodes.extend([
            ...
        ])

        mesh_counter += 2

    else:
        gltf_export.nodes.extend([
            pygltflib.Node(
                mesh = visited[mesh],
                rotation = node.rotation,
                scale = node.scale,
                name = node.name,
                children = [
                    pygltflib.Node(
                        mesh = visited[mesh] + 1,
                        rotation = node.rotation,
                        scale = node.scale,
                        name = node.name
                    )
                ]
            )
        ])
'''

'\nvisited = {}\n\nfor node in gltf.nodes:\n    mesh = node.mesh\n\n    if mesh not in visited:\n        visited[mesh] = mesh_counter\n\n        gltf_export.nodes.extend([\n            ...\n        ])\n\n        mesh_counter += 2\n\n    else:\n        gltf_export.nodes.extend([\n            pygltflib.Node(\n                mesh = visited[mesh],\n                rotation = node.rotation,\n                scale = node.scale,\n                name = node.name,\n                children = [\n                    pygltflib.Node(\n                        mesh = visited[mesh] + 1,\n                        rotation = node.rotation,\n                        scale = node.scale,\n                        name = node.name\n                    )\n                ]\n            )\n        ])\n'

In theory now, we should have a running count of individual meshes in the scene. We iterate mesh_counter by 2 every time we create a new mesh in the scene. If the mesh has been seen before, we use the dictionary `visited` to keep track of the mesh_id that corresponds to that mesh.

Now, we need to also focus on creating our low-res counterparts. This will involve extending the `meshes` in the scene, and will only be run if the mesh is not visited before.

In [126]:
# Pseduo code
## Meshes

'''
visited = {}

for node in gltf.nodes:
    mesh = node.mesh

    if mesh not in visited:
        visited[mesh] = mesh_counter

        gltf_export.nodes.extend([
            ...
        ])
        
        gltf_export.meshes.extend([
            pygltflib.Mesh(
                primitives=[
                    pygltflib.Primitive(
                        attributes=pygltflib.Attributes(
                            POSITION = accessor_counter + 1, 
                            NORMAL = accessor_counter + 2,
                        ),
                        indices = accessor_counter + 0
                    )
                ]
            ),
            pygltflib.Mesh(
                primitives=[
                    pygltflib.Primitive(
                        attributes=pygltflib.Attributes(
                            POSITION = accessor_counter + 4, 
                            NORMAL = accessor_counter + 5,
                        ),
                        indices = accessor_counter + 3
                    )
                ]
            )
        ])

        mesh_counter += 2
        accessor_counter += 6

    else:
        gltf_export.nodes.extend([
            ...
        ])

'''

'\nvisited = {}\n\nfor node in gltf.nodes:\n    mesh = node.mesh\n\n    if mesh not in visited:\n        visited[mesh] = mesh_counter\n\n        gltf_export.nodes.extend([\n            ...\n        ])\n\n        gltf_export.meshes.extend([\n            pygltflib.Mesh(\n                primitives=[\n                    pygltflib.Primitive(\n                        attributes=pygltflib.Attributes(\n                            POSITION = accessor_counter + 1, \n                            NORMAL = accessor_counter + 2,\n                        ),\n                        indices = accessor_counter + 0\n                    )\n                ]\n            ),\n            pygltflib.Mesh(\n                primitives=[\n                    pygltflib.Primitive(\n                        attributes=pygltflib.Attributes(\n                            POSITION = accessor_counter + 4, \n                            NORMAL = accessor_counter + 5,\n                        ),\n                        i

In the code above, we first create 2 new meshes in the scene if the mesh is new. This will be the high and low resolution versions of the mesh respectively, and will be added to our nodes as parent and child.

Below meshes we have acessors. These tell us what the data is for each POSITION, NORMAL and INDICES attribute of our mesh.

In [127]:
# Pseudo Code
## Accessors

'''
visited = {}

for node in gltf.nodes:
    mesh = node.mesh

    if mesh not in visited:
        visited[mesh] = mesh_counter

        gltf_export.nodes.extend([
            ...
        ])

        gltf_export.meshes.extend([
            ...
        ])

        gltf_export.accessors.extend([
            
            # Indices - Original Remapped
            pygltflib.Accessor(
                bufferView = bufferView_counter
                componentType = pygltflib.UNSIGNED_BYTE,
                count = f_matrix_rmp.size,
                type = pygltflib.SCALAR,
                max = [int(f_matrix_rmp.max())],
                min = [int(f_matrix_rmp.min())],
            ),

            # Vertices - Original Remapped
            pygltflib.Accessor(
                bufferView = bufferView_counter + 1
                componentType = pygltflib.FLOAT,
                count = len(v_matrix_rmp),
                type = pygltflib.VEC3,
                max = v_matrix_rmp.max(axis=0).tolist(),
                min = v_matrix_rmp.min(axis=0).tolist(),
            ),

            # Normals - Original Remapped
            pygltflib.Accessor(
                bufferView = bufferView_counter + 2
                byteOffset = 0,
                componentType = pygltflib.FLOAT,
                count = len(n_matrix_rmp),
                type = pygltflib.VEC3,
                max = n_matrix_rmp.max(axis=0).tolist(),
                min = n_matrix_rmp.min(axis=0).tolist(),
            ),

            # Indices - Decimated
            pygltflib.Accessor(
                bufferView = bufferView_counter + 3
                componentType = pygltflib.UNSIGNED_BYTE,
                count = f_matrix_dm.size,
                type = pygltflib.SCALAR,
                max = [int(f_matrix_dm.max())],
                min = [int(f_matrix_dm.min())],
            ),

            # Vertices - Decimated
            pygltflib.Accessor(
                bufferView = bufferView_counter + 4
                componentType = pygltflib.FLOAT,
                count = len(v_matrix_dm),
                type = pygltflib.VEC3,
                max = v_matrix_dm.max(axis=0).tolist(),
                min = v_matrix_dm.min(axis=0).tolist(),
            ),

            # Normals - Decimated
            pygltflib.Accessor(
                bufferView = bufferView_counter + 5
                byteOffset = 0,
                componentType = pygltflib.FLOAT,
                count = len(n_matrix_dm),
                type = pygltflib.VEC3,
                max = n_matrix_dm.max(axis=0).tolist(),
                min = n_matrix_dm.min(axis=0).tolist(),
            )
        ])

        mesh_counter += 2
        accessor_counter += 6
        bufferView_counter += 6

    else:
        gltf_export.nodes.extend([
            ...
        ])

'''

'\nvisited = {}\n\nfor node in gltf.nodes:\n    mesh = node.mesh\n\n    if mesh not in visited:\n        visited[mesh] = mesh_counter\n\n        gltf_export.nodes.extend([\n            ...\n        ])\n\n        gltf_export.meshes.extend([\n            ...\n        ])\n\n        gltf_export.accessors.extend([\n\n            # Indices - Original Remapped\n            pygltflib.Accessor(\n                bufferView = bufferView_counter\n                componentType = pygltflib.UNSIGNED_BYTE,\n                count = f_matrix_rmp.size,\n                type = pygltflib.SCALAR,\n                max = [int(f_matrix_rmp.max())],\n                min = [int(f_matrix_rmp.min())],\n            ),\n\n            # Vertices - Original Remapped\n            pygltflib.Accessor(\n                bufferView = bufferView_counter + 1\n                componentType = pygltflib.FLOAT,\n                count = len(v_matrix_rmp),\n                type = pygltflib.VEC3,\n                max = v_matrix_rmp.

Now, below accessors, we have bufferviews.

In [128]:
# Pseudo code
## BufferViews

'''
visited = {}

for node in gltf.nodes:
    mesh = node.mesh

    if mesh not in visited:
        visited[mesh] = mesh_counter

        gltf_export.nodes.extend([
            ...
        ])

        gltf_export.meshes.extend([
            ...
        ])

        gltf_export.accessors.extend([
            ...
        ])

        gltf_export.bufferViews.extend([
            
            # Indices - Remapped
            pygltflib.BufferView(
                buffer=0,
                byteOffset = currByteOffset,
                byteLength = len(triangles_rmp_binary_blob),
                target=pygltflib.ELEMENT_ARRAY_BUFFER
            ),

            # Vertices - Remapped
            pygltflib.BufferView(
                buffer=0,
                byteOffset= currByteOffset + len(triangles_rmp_binary_blob),
                byteLength = len(points_rmp_binary_blob),
                target=pygltflib.ARRAY_BUFFER
            ),

            # Normals - Remapped
            pygltflib.BufferView(
                buffer=0,
                byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
                byteLength = len(normals_rmp_binary_blob),
                target=pygltflib.ARRAY_BUFFER
            ),

            # Indices - Decimated
            pygltflib.BufferView(
                buffer=0,
                byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob),
                byteLength = len(triangles_dm_binary_blob),
                target=pygltflib.ELEMENT_ARRAY_BUFFER
            ),

            # Vertices - Decimated
            pygltflib.BufferView(
                buffer=0,
                byteOffset= currByteOffset + len(triangles_rmp_binary_blob),
                byteLength = len(points_dm_binary_blob),
                target=pygltflib.ARRAY_BUFFER
            ),

            # Normals - Decimated
            pygltflib.BufferView(
                buffer=0,
                byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
                byteLength = len(normals_dm_binary_blob),
                target=pygltflib.ARRAY_BUFFER
            ),

        ])

        mesh_counter += 2
        accessor_counter += 6
        bufferView_counter += 6
        currByteOffset += len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_dm_binary_blob) + len(triangles_dm_binary_blob)

    else:
        gltf_export.nodes.extend([
            ...
        ])
'''

'\nvisited = {}\n\nfor node in gltf.nodes:\n    mesh = node.mesh\n\n    if mesh not in visited:\n        visited[mesh] = mesh_counter\n\n        gltf_export.nodes.extend([\n            ...\n        ])\n\n        gltf_export.meshes.extend([\n            ...\n        ])\n\n        gltf_export.accessors.extend([\n            ...\n        ])\n\n        gltf_export.bufferViews.extend([\n\n            # Indices - Remapped\n            pygltflib.BufferView(\n                buffer=0,\n                byteOffset = currByteOffset,\n                byteLength = len(triangles_rmp_binary_blob),\n                target=pygltflib.ELEMENT_ARRAY_BUFFER\n            ),\n\n            # Vertices - Remapped\n            pygltflib.BufferView(\n                buffer=0,\n                byteOffset= currByteOffset + len(triangles_rmp_binary_blob),\n                byteLength = len(points_rmp_binary_blob),\n                target=pygltflib.ARRAY_BUFFER\n            ),\n\n            # Normals - Remapped\n   

The code block above represents the careful open heart surgery we're conducting on the gltf file. We create 6 new bufferviews in our scene. The first 3 correspond to our remapped original hi-res mesh and includes the index array, verex array and the normal array in that order. The next 3 correspond to our decimated mesh and include the decimated triangle array, the subset of the remapped vertex array and the subset of the remapped normals array.

Lastly, the buffer will need to be updated with the additional data.

In [129]:
# Pseudo code
## Buffer

'''
visited = {}

for node in gltf.nodes:
    mesh = node.mesh

    if mesh not in visited:
        visited[mesh] = mesh_counter

        gltf_export.nodes.extend([
            ...
        ])

        gltf_export.meshes.extend([
            ...
        ])

        gltf_export.accessors.extend([
            ...
        ])

        gltf_export.bufferViews.extend([
            ...
        ])

        main_binary_blob += triangles_rmp_binary_blob + points_rmp_binary_blob + normals_rmp_binary_blob + triangles_dm_binary_blob

        mesh_counter += 2
        accessor_counter += 6
        bufferView_counter += 6
        currByteOffset += len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob) + len(triangles_dm_binary_blob)

    else:
        gltf_export.nodes.extend([
            ...
        ])
'''

'\nvisited = {}\n\nfor node in gltf.nodes:\n    mesh = node.mesh\n\n    if mesh not in visited:\n        visited[mesh] = mesh_counter\n\n        gltf_export.nodes.extend([\n            ...\n        ])\n\n        gltf_export.meshes.extend([\n            ...\n        ])\n\n        gltf_export.accessors.extend([\n            ...\n        ])\n\n        gltf_export.bufferViews.extend([\n            ...\n        ])\n\n        main_binary_blob += triangles_rmp_binary_blob + points_rmp_binary_blob + normals_rmp_binary_blob + triangles_dm_binary_blob\n\n        mesh_counter += 2\n        accessor_counter += 6\n        bufferView_counter += 6\n        currByteOffset += len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob) + len(triangles_dm_binary_blob)\n\n    else:\n        gltf_export.nodes.extend([\n            ...\n        ])\n'

This is going to be a long one. Let's strap in.

First, we need to append our `decimate_mesh` script to also remap the normals.

In [130]:
def decimate_mesh_2(v_mat, f_mat, n_mat, perc_red=0.0):
    """
    
    """
    
    ms = pymeshlab.MeshSet()
    org_m = pymeshlab.Mesh(v_mat, f_mat)

    ms.add_mesh(org_m)

    ms.meshing_decimation_quadric_edge_collapse(optimalplacement=False, targetperc=perc_red) #preservetopology=True, qualitythr=0.5
    ms.compute_normal_per_vertex()

    m = ms.current_mesh()

    v_dm = m.vertex_matrix()
    f_dm = m.face_matrix()
    n_dm = m.vertex_normal_matrix()

    v_dict = { tuple(row): i for i, row in enumerate(v_dm) }
    v_remapping = np.argsort(np.array([v_dict.get(tuple(row), np.inf) for row in v_mat ]))

    v_org_rmp = v_mat[v_remapping]
    n_org_rmp = n_mat[v_remapping] # Remap the normals according to the remapped vertex matrix

    v_inv_mapping = np.argsort(v_remapping)
    f_org_rmp = v_inv_mapping[f_mat]

    ms.clear()

    return v_dm, f_dm, n_dm, v_org_rmp, f_org_rmp, n_org_rmp

Let's also create some helper functions to save us from a wall of text script

In [131]:
GLTF_COMPONENT_TYPES = {
    5120: np.int8,
    5121: np.uint8,
    5122: np.int16,
    5123: np.uint16,
    5125: np.uint32,
    5126: np.float32
}

In [ ]:
def get_triangles_array(in_gltf, mesh):  
    binary_blob = in_gltf.binary_blob()
    
    triangles_accessor = in_gltf.accessors[in_gltf.meshes[mesh].primitives[0].indices]
    triangles_buffer_view = in_gltf.bufferViews[triangles_accessor.bufferView]
    triangles_array = np.frombuffer(
        binary_blob[
            triangles_buffer_view.byteOffset
            + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
            + triangles_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[triangles_accessor.componentType],
        count=triangles_accessor.count,
    ).reshape((-1, 3))

    return triangles_array

def get_points_array(in_gltf, mesh):
    binary_blob = in_gltf.binary_blob()
    
    points_accessor = in_gltf.accessors[in_gltf.meshes[mesh].primitives[0].attributes.POSITION]
    points_buffer_view = in_gltf.bufferViews[points_accessor.bufferView]
    points_array = np.frombuffer(
        binary_blob[
            points_buffer_view.byteOffset
            + points_accessor.byteOffset : points_buffer_view.byteOffset
            + points_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[points_accessor.componentType],
        count=points_accessor.count * 3,
    ).reshape((-1, 3))

    return points_array

def get_normals_array(in_gltf, mesh):
    binary_blob = in_gltf.binary_blob()
    
    normals_accessor = in_gltf.accessors[in_gltf.meshes[mesh].primitives[0].attributes.NORMAL]
    normals_buffer_view = in_gltf.bufferViews[normals_accessor.bufferView]
    normals_array = np.frombuffer(
        binary_blob[
            normals_buffer_view.byteOffset
            + normals_accessor.byteOffset : normals_buffer_view.byteOffset
            + normals_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[normals_accessor.componentType],
        count=normals_accessor.count * 3,
    ).reshape((-1, 3))

    return normals_array

Now, lets build our main loop.

In [133]:
# Initialize empty GLTF container to store the newly created mesh objects.
gltf_export = pygltflib.GLTF2()
gltf_export.scenes.append(pygltflib.Scene(nodes=[]))
gltf_export.scene = 0

In [134]:
# Initialize Counters and Binary Blob
node_counter = 0
mesh_counter = 0
accessor_counter = 0
bufferView_counter = 0
currByteOffset = 0

main_binary_blob = bytearray()

In [135]:
visited = {}

for node in gltf.nodes:

    mesh = node.mesh
    print(f"working on {mesh}")

    if mesh in visited:
        # If mesh exists, use the existing mesh ID. Do not need to create new bufferviews etc.
        ## Decimated comes SECOND
        print("exists")
        gltf_export.nodes.extend([
            pygltflib.Node(
                mesh = visited[mesh],
                rotation = node.rotation,
                translation = node.translation,
                matrix = node.matrix,
                scale = node.scale,
                name = node.name,
                children = [ node_counter + 1 ]
            ),
            pygltflib.Node(
                mesh = visited[mesh] + 1,
                # rotation = node.rotation,
                # translation = node.translation,
                # matrix = node.matrix,
                # scale = node.scale,
                name = node.name
            )
        ])
        gltf_export.scenes[0].nodes.extend([node_counter])
        node_counter += 2

        continue

    # If mesh does not exist yet, first add to dictionary
    print("does not exist need to add")
    visited[mesh] = mesh_counter

    # Get the original index, position and normal matrices
    f_org = get_triangles_array(gltf, mesh)
    v_org = get_points_array(gltf, mesh)
    n_org = get_normals_array(gltf, mesh)
    print("matrices acquired")

    # Apply the decimation and remapping
    v_dm, f_dm, n_dm, v_org_rmp, f_org_rmp, n_org_rmp = decimate_mesh_2(v_org, f_org, n_org, 0.6)
    print("decimation done")

    f_org_rmp = f_org_rmp.astype(np.uint16)
    f_dm = f_dm.astype(np.uint16)

    v_org_rmp = v_org_rmp.astype(np.float32)
    v_dm = v_dm.astype(np.float32)

    n_org_rmp = n_org_rmp.astype(np.float32)
    n_dm = n_dm.astype(np.float32)

    # Convert to Bytes
    triangles_rmp_binary_blob = f_org_rmp.flatten().tobytes()
    points_rmp_binary_blob = v_org_rmp.tobytes()
    normals_rmp_binary_blob = n_org_rmp.tobytes()
    triangles_dm_binary_blob = f_dm.flatten().tobytes()
    points_dm_binary_blob = v_dm.tobytes()
    normals_dm_binary_blob = n_dm.tobytes()
    print("converted to bytes")

    # Append to Buffer
    main_binary_blob.extend(triangles_rmp_binary_blob + points_rmp_binary_blob + normals_rmp_binary_blob + triangles_dm_binary_blob)
    print("appended buffer")

    # Create New BufferViews
    gltf_export.bufferViews.extend([
                
        ## Indices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset,
            byteLength = len(triangles_rmp_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Indices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob),
            byteLength = len(triangles_dm_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset= currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),
    ])
    print("created new bufferviews")

    # Create new Accessors
    gltf_export.accessors.extend([
                
        ## Indices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_org_rmp.size,
            type = pygltflib.SCALAR,
            max = [int(f_org_rmp.max())],
            min = [int(f_org_rmp.min())],
        ),

        ## Vertices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 1,
            componentType = pygltflib.FLOAT,
            count = len(v_org_rmp),
            type = pygltflib.VEC3,
            max = v_org_rmp.max(axis=0).tolist(),
            min = v_org_rmp.min(axis=0).tolist(),
        ),

        ## Normals - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 2,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_org_rmp),
            type = pygltflib.VEC3,
            max = n_org_rmp.max(axis=0).tolist(),
            min = n_org_rmp.min(axis=0).tolist(),
        ),

        ## Indices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 3,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_dm.size,
            type = pygltflib.SCALAR,
            max = [int(f_dm.max())],
            min = [int(f_dm.min())],
        ),

        ## Vertices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 4,
            componentType = pygltflib.FLOAT,
            count = len(v_dm),
            type = pygltflib.VEC3,
            max = v_dm.max(axis=0).tolist(),
            min = v_dm.min(axis=0).tolist(),
        ),

        ## Normals - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 5,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_dm),
            type = pygltflib.VEC3,
            max = n_dm.max(axis=0).tolist(),
            min = n_dm.min(axis=0).tolist(),
        )
    ])
    print("created new accessors")

    # Create new Meshes
    gltf_export.meshes.extend([
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 1, 
                        NORMAL = accessor_counter + 2,
                    ),
                    indices = accessor_counter + 0
                )
            ]
        ),
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 4, 
                        NORMAL = accessor_counter + 5,
                    ),
                    indices = accessor_counter + 3
                )
            ]
        )
    ])
    print("created new meshes")

    # Create new Nodes
    gltf_export.nodes.extend([
        pygltflib.Node(
            mesh = mesh_counter,
            rotation = node.rotation,
            translation = node.translation,
            scale = node.scale,
            matrix = node.matrix,
            name = node.name,
            children = [ node_counter + 1]
        ),
        pygltflib.Node(
            mesh = mesh_counter + 1,
            # rotation = node.rotation,
            # translation = node.translation,
            # scale = node.scale,
            # matrix = node.matrix,
            name = node.name
        )
    ])
    print("created new nodes")

    # Link nodes to scene
    gltf_export.scenes[0].nodes.extend([node_counter])

    # Increment all our counters
    node_counter += 2
    mesh_counter += 2
    accessor_counter += 6
    bufferView_counter += 6
    currByteOffset += len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob) + len(triangles_dm_binary_blob)
    
    print("increment successful")
    print("run successful")
    

working on 0
does not exist need to add
matrices acquired
decimation done
converted to bytes
appended buffer
created new bufferviews
created new accessors
created new meshes
created new nodes
increment successful
run successful
working on 0
exists
working on 1
does not exist need to add
matrices acquired
decimation done
converted to bytes
appended buffer
created new bufferviews
created new accessors
created new meshes
created new nodes
increment successful
run successful
working on 1
exists
working on 1
exists
working on 1
exists
working on 1
exists
working on 1
exists
working on 1
exists
working on 2
does not exist need to add
matrices acquired
decimation done
converted to bytes
appended buffer
created new bufferviews
created new accessors
created new meshes
created new nodes
increment successful
run successful
working on 2
exists
working on 2
exists
working on 2
exists
working on 2
exists
working on 2
exists
working on 2
exists
working on 3
does not exist need to add
matrices acquire

The last step will be to add our `main_binary_blob` buffer to the file.

In [136]:
gltf_export.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
gltf_export.set_binary_blob(main_binary_blob)

# export_filename = "mep-fittings_hi-lo-res_ifc-guid.glb"
# gltf_export.save(export_filename)

We seem to be running into crashed kernels. Probably best to export this out to a script and run it from the terminal.

We have seen dead kernels before this, and attributed it to improper typing in the `indices` array. This needs to be sufficiently large enough to account for all the vertex numbers. Usually `uint16` does the job. When we set the type of this array explicitly, our code runs. However, on load we observe jagged edges in our meshes. We have seen before that this corresponds to improper indexing being conducted at the mesh level.

![Jagged Edges observed on LOD creation](../img/jagged-edges-on-lod-creation.png)

Looking within the metadata of the object, we see that this is called a gully trap. We export this out as a standalone object for testing.

In [137]:
gltf_gully = pygltflib.GLTF2().load('data/sixty5-gullytrap-example.glb')
gltf_gully.nodes

[Node(extensions={}, extras={}, mesh=0, skin=None, rotation=None, translation=[11.01897144317627, 23.826234817504883, -10.733771324157715], scale=None, children=[], matrix=None, camera=None, name='IfcFlowTerminal/LOROX serie H2:4016:3173913')]

Let's try running this test object through our algorithm above and see what we get.

In [138]:
# Initialize empty GLTF container to store the newly created mesh objects.
gltf_test = pygltflib.GLTF2()
gltf_test.scenes.append(pygltflib.Scene(nodes=[]))
gltf_test.scene = 0

# Initialize Counters and Binary Blob
node_counter = 0
mesh_counter = 0
accessor_counter = 0
bufferView_counter = 0
currByteOffset = 0

main_binary_blob = bytearray()

visited = {}

for node in gltf_gully.nodes:

    mesh = node.mesh
    print(f"working on {mesh}")

    if mesh in visited:
        # If mesh exists, use the existing mesh ID. Do not need to create new bufferviews etc.
        ## Decimated comes SECOND
        print("exists")
        gltf_test.nodes.extend([
            pygltflib.Node(
                mesh = visited[mesh],
                rotation = node.rotation,
                translation = node.translation,
                matrix = node.matrix,
                scale = node.scale,
                name = node.name,
                children = [ node_counter + 1 ]
            ),
            pygltflib.Node(
                mesh = visited[mesh] + 1,
                name = node.name
            )
        ])
        gltf_test.scenes[0].nodes.extend([node_counter])
        node_counter += 2

        continue

    # If mesh does not exist yet, first add to dictionary
    print("does not exist need to add")
    visited[mesh] = mesh_counter

    # Get the original index, position and normal matrices
    f_org = get_triangles_array(gltf_gully, mesh)
    v_org = get_points_array(gltf_gully, mesh)
    n_org = get_normals_array(gltf_gully, mesh)
    print("matrices acquired")

    # Apply the decimation and remapping
    v_dm, f_dm, n_dm, v_org_rmp, f_org_rmp, n_org_rmp = decimate_mesh_2(v_org, f_org, n_org)
    print("decimation done")

    f_org_rmp = f_org_rmp.astype(np.uint16)
    f_dm = f_dm.astype(np.uint16)

    v_org_rmp = v_org_rmp.astype(np.float32)
    v_dm = v_dm.astype(np.float32)

    n_org_rmp = n_org_rmp.astype(np.float32)
    n_dm = n_dm.astype(np.float32)

    # Convert to Bytes
    triangles_rmp_binary_blob = f_org_rmp.flatten().tobytes()
    points_rmp_binary_blob = v_org_rmp.tobytes()
    normals_rmp_binary_blob = n_org_rmp.tobytes()
    triangles_dm_binary_blob = f_dm.flatten().tobytes()
    points_dm_binary_blob = v_dm.tobytes()
    normals_dm_binary_blob = n_dm.tobytes()
    print("converted to bytes")

    # Append to Buffer
    main_binary_blob.extend(triangles_rmp_binary_blob + points_rmp_binary_blob + normals_rmp_binary_blob + triangles_dm_binary_blob)
    print("appended buffer")

    # Create New BufferViews
    gltf_test.bufferViews.extend([
                
        ## Indices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset,
            byteLength = len(triangles_rmp_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Indices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob),
            byteLength = len(triangles_dm_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset= currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),
    ])
    print("created new bufferviews")

    # Create new Accessors
    gltf_test.accessors.extend([
                
        ## Indices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_org_rmp.size,
            type = pygltflib.SCALAR,
            max = [int(f_org_rmp.max())],
            min = [int(f_org_rmp.min())],
        ),

        ## Vertices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 1,
            componentType = pygltflib.FLOAT,
            count = len(v_org_rmp),
            type = pygltflib.VEC3,
            max = v_org_rmp.max(axis=0).tolist(),
            min = v_org_rmp.min(axis=0).tolist(),
        ),

        ## Normals - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 2,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_org_rmp),
            type = pygltflib.VEC3,
            max = n_org_rmp.max(axis=0).tolist(),
            min = n_org_rmp.min(axis=0).tolist(),
        ),

        ## Indices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 3,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_dm.size,
            type = pygltflib.SCALAR,
            max = [int(f_dm.max())],
            min = [int(f_dm.min())],
        ),

        ## Vertices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 4,
            componentType = pygltflib.FLOAT,
            count = len(v_dm),
            type = pygltflib.VEC3,
            max = v_dm.max(axis=0).tolist(),
            min = v_dm.min(axis=0).tolist(),
        ),

        ## Normals - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 5,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_dm),
            type = pygltflib.VEC3,
            max = n_dm.max(axis=0).tolist(),
            min = n_dm.min(axis=0).tolist(),
        )
    ])
    print("created new accessors")

    # Create new Meshes
    gltf_test.meshes.extend([
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 1, 
                        NORMAL = accessor_counter + 2,
                    ),
                    indices = accessor_counter + 0
                )
            ]
        ),
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 4, 
                        NORMAL = accessor_counter + 5,
                    ),
                    indices = accessor_counter + 3
                )
            ]
        )
    ])
    print("created new meshes")

    # Create new Nodes
    gltf_test.nodes.extend([
        pygltflib.Node(
            mesh = mesh_counter,
            rotation = node.rotation,
            translation = node.translation,
            scale = node.scale,
            matrix = node.matrix,
            name = node.name,
            children = [ node_counter + 1]
        ),
        pygltflib.Node(
            mesh = mesh_counter + 1,
            name = node.name
        )
    ])
    print("created new nodes")

    # Link nodes to scene
    gltf_test.scenes[0].nodes.extend([node_counter])

    # Increment all our counters
    node_counter += 2
    mesh_counter += 2
    accessor_counter += 6
    bufferView_counter += 6
    currByteOffset += len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob) + len(triangles_dm_binary_blob)
    
    print("increment successful")
    print("run successful")
    

working on 0
does not exist need to add
matrices acquired
decimation done
converted to bytes
appended buffer
created new bufferviews
created new accessors
created new meshes
created new nodes
increment successful
run successful


In [139]:
# gltf_test.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
# gltf_test.set_binary_blob(main_binary_blob)

# export_filename = "mep-gulley-trap-test.glb"
# gltf_test.save(export_filename)

Still the reindexing issue shows up. Here is the decimate mesh function we have been using.

In [140]:
def decimate_mesh_2(v_mat, f_mat, n_mat, perc_red=0.0):
    """
    
    """
    
    ms = pymeshlab.MeshSet()
    org_m = pymeshlab.Mesh(v_mat, f_mat)

    ms.add_mesh(org_m)

    ms.meshing_decimation_quadric_edge_collapse(optimalplacement=False, targetperc=perc_red) #preservetopology=True, qualitythr=0.5
    ms.compute_normal_per_vertex()

    m = ms.current_mesh()

    v_dm = m.vertex_matrix()
    f_dm = m.face_matrix()
    n_dm = m.vertex_normal_matrix()

    v_dict = { tuple(row): i for i, row in enumerate(v_dm) }
    v_remapping = np.argsort(np.array([v_dict.get(tuple(row), np.inf) for row in v_mat ]))

    v_org_rmp = v_mat[v_remapping]
    n_org_rmp = n_mat[v_remapping] # Remap the normals according to the remapped vertex matrix

    v_inv_mapping = np.argsort(v_remapping)
    f_org_rmp = v_inv_mapping[f_mat]

    ms.clear()

    return v_dm, f_dm, n_dm, v_org_rmp, f_org_rmp, n_org_rmp

Let's see what these matrices look like on their own.

In [141]:
get_triangles_array(gltf_gully, 0)

array([[     2,      9,     17],
       [    21,     26,     37],
       [    40,     45,     57],
       ...,
       [102213, 102202, 102199],
       [102251, 102253, 102244],
       [102254, 102241, 102237]], shape=(38841, 3), dtype=uint32)

In [142]:
get_points_array(gltf_gully, 0)

array([[-0.0315548 , -0.29303598,  0.02671323],
       [-0.0315548 , -0.29303598,  0.02671323],
       [-0.0315548 , -0.29303598,  0.02671323],
       ...,
       [ 0.23906961, -0.28738472, -0.035     ],
       [ 0.23906961, -0.28738472, -0.035     ],
       [ 0.23906961, -0.28738472, -0.035     ]],
      shape=(102265, 3), dtype=float32)

This single object has apparently got 102k vertices... that seems... wrong.

In [143]:
gltf_gully.meshes[0]

Mesh(extensions={}, extras={}, primitives=[Primitive(extensions={}, extras={}, attributes=Attributes(POSITION=0, NORMAL=1, TANGENT=None, TEXCOORD_0=None, TEXCOORD_1=None, COLOR_0=None, JOINTS_0=None, WEIGHTS_0=None), indices=2, mode=4, material=None, targets=[])], weights=[], name='102/2233328')

In [144]:
binary_blob = gltf_gully.binary_blob()
    
triangles_accessor = gltf.accessors[gltf.meshes[mesh].primitives[0].indices]
triangles_buffer_view = gltf.bufferViews[triangles_accessor.bufferView]
triangles_array = np.frombuffer(
    binary_blob[
        triangles_buffer_view.byteOffset
        + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
        + triangles_buffer_view.byteLength
    ],
    dtype=GLTF_COMPONENT_TYPES[triangles_accessor.componentType],
    count=triangles_accessor.count,
).reshape((-1, 3))
    
points_accessor = gltf.accessors[gltf.meshes[mesh].primitives[0].attributes.POSITION]
points_buffer_view = gltf.bufferViews[points_accessor.bufferView]
points_array = np.frombuffer(
    binary_blob[
        points_buffer_view.byteOffset
        + points_accessor.byteOffset : points_buffer_view.byteOffset
        + points_buffer_view.byteLength
    ],
    dtype=GLTF_COMPONENT_TYPES[points_accessor.componentType],
    count=points_accessor.count * 3,
).reshape((-1, 3))

triangles_array.size

408

In [145]:
points_array.size

690

That seems more like it.

In [146]:
get_triangles_array(gltf_gully, 0)

array([[     2,      9,     17],
       [    21,     26,     37],
       [    40,     45,     57],
       ...,
       [102213, 102202, 102199],
       [102251, 102253, 102244],
       [102254, 102241, 102237]], shape=(38841, 3), dtype=uint32)

For some reason these functions are not working as intended. If we hard code them within our main function code, do they work? Let's see.

In [147]:
# Initialize empty GLTF container to store the newly created mesh objects.
gltf_test = pygltflib.GLTF2()
gltf_test.scenes.append(pygltflib.Scene(nodes=[]))
gltf_test.scene = 0

# Initialize Counters and Binary Blob
node_counter = 0
mesh_counter = 0
accessor_counter = 0
bufferView_counter = 0
currByteOffset = 0

main_binary_blob = bytearray()
binary_blob = gltf_gully.binary_blob()

visited = {}

for node in gltf_gully.nodes:

    mesh = node.mesh
    print(f"working on {mesh}")

    if mesh in visited:
        # If mesh exists, use the existing mesh ID. Do not need to create new bufferviews etc.
        ## Decimated comes SECOND
        print("exists")
        gltf_test.nodes.extend([
            pygltflib.Node(
                mesh = visited[mesh],
                rotation = node.rotation,
                translation = node.translation,
                matrix = node.matrix,
                scale = node.scale,
                name = node.name,
                children = [ node_counter + 1 ]
            ),
            pygltflib.Node(
                mesh = visited[mesh] + 1,
                name = node.name
            )
        ])
        gltf_test.scenes[0].nodes.extend([node_counter])
        node_counter += 2

        continue

    # If mesh does not exist yet, first add to dictionary
    print("does not exist need to add")
    visited[mesh] = mesh_counter

    # Get the original index, position and normal matrices
    triangles_accessor = gltf_gully.accessors[gltf_gully.meshes[mesh].primitives[0].indices]
    triangles_buffer_view = gltf_gully.bufferViews[triangles_accessor.bufferView]
    f_org = np.frombuffer(
        binary_blob[
            triangles_buffer_view.byteOffset
            + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
            + triangles_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[triangles_accessor.componentType],
        count=triangles_accessor.count,
    ).reshape((-1, 3))

    points_accessor = gltf_gully.accessors[gltf_gully.meshes[mesh].primitives[0].attributes.POSITION]
    points_buffer_view = gltf_gully.bufferViews[points_accessor.bufferView]
    v_org = np.frombuffer(
        binary_blob[
            points_buffer_view.byteOffset
            + points_accessor.byteOffset : points_buffer_view.byteOffset
            + points_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[points_accessor.componentType],
        count=points_accessor.count * 3,
    ).reshape((-1, 3))

    normals_accessor = gltf_gully.accessors[gltf_gully.meshes[mesh].primitives[0].attributes.NORMAL]
    normals_buffer_view = gltf_gully.bufferViews[normals_accessor.bufferView]
    n_org = np.frombuffer(
        binary_blob[
            normals_buffer_view.byteOffset
            + normals_accessor.byteOffset : normals_buffer_view.byteOffset
            + normals_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[normals_accessor.componentType],
        count=normals_accessor.count * 3,
    ).reshape((-1, 3))
    
    print("matrices acquired")

    # Apply the decimation and remapping
    v_dm, f_dm, n_dm, v_org_rmp, f_org_rmp, n_org_rmp = decimate_mesh_2(v_org, f_org, n_org)
    print("decimation done")

    f_org_rmp = f_org_rmp.astype(np.uint16)
    f_dm = f_dm.astype(np.uint16)

    v_org_rmp = v_org_rmp.astype(np.float32)
    v_dm = v_dm.astype(np.float32)

    n_org_rmp = n_org_rmp.astype(np.float32)
    n_dm = n_dm.astype(np.float32)

    # Convert to Bytes
    triangles_rmp_binary_blob = f_org_rmp.flatten().tobytes()
    points_rmp_binary_blob = v_org_rmp.tobytes()
    normals_rmp_binary_blob = n_org_rmp.tobytes()
    triangles_dm_binary_blob = f_dm.flatten().tobytes()
    points_dm_binary_blob = v_dm.tobytes()
    normals_dm_binary_blob = n_dm.tobytes()
    print("converted to bytes")

    # Append to Buffer
    main_binary_blob.extend(triangles_rmp_binary_blob + points_rmp_binary_blob + normals_rmp_binary_blob + triangles_dm_binary_blob)
    print("appended buffer")

    # Create New BufferViews
    gltf_test.bufferViews.extend([
                
        ## Indices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset,
            byteLength = len(triangles_rmp_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Indices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob),
            byteLength = len(triangles_dm_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset= currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),
    ])
    print("created new bufferviews")

    # Create new Accessors
    gltf_test.accessors.extend([
                
        ## Indices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_org_rmp.size,
            type = pygltflib.SCALAR,
            max = [int(f_org_rmp.max())],
            min = [int(f_org_rmp.min())],
        ),

        ## Vertices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 1,
            componentType = pygltflib.FLOAT,
            count = len(v_org_rmp),
            type = pygltflib.VEC3,
            max = v_org_rmp.max(axis=0).tolist(),
            min = v_org_rmp.min(axis=0).tolist(),
        ),

        ## Normals - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 2,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_org_rmp),
            type = pygltflib.VEC3,
            max = n_org_rmp.max(axis=0).tolist(),
            min = n_org_rmp.min(axis=0).tolist(),
        ),

        ## Indices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 3,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_dm.size,
            type = pygltflib.SCALAR,
            max = [int(f_dm.max())],
            min = [int(f_dm.min())],
        ),

        ## Vertices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 4,
            componentType = pygltflib.FLOAT,
            count = len(v_dm),
            type = pygltflib.VEC3,
            max = v_dm.max(axis=0).tolist(),
            min = v_dm.min(axis=0).tolist(),
        ),

        ## Normals - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 5,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_dm),
            type = pygltflib.VEC3,
            max = n_dm.max(axis=0).tolist(),
            min = n_dm.min(axis=0).tolist(),
        )
    ])
    print("created new accessors")

    # Create new Meshes
    gltf_test.meshes.extend([
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 1, 
                        NORMAL = accessor_counter + 2,
                    ),
                    indices = accessor_counter + 0
                )
            ]
        ),
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 4, 
                        NORMAL = accessor_counter + 5,
                    ),
                    indices = accessor_counter + 3
                )
            ]
        )
    ])
    print("created new meshes")

    # Create new Nodes
    gltf_test.nodes.extend([
        pygltflib.Node(
            mesh = mesh_counter,
            rotation = node.rotation,
            translation = node.translation,
            scale = node.scale,
            matrix = node.matrix,
            name = node.name,
            children = [ node_counter + 1]
        ),
        pygltflib.Node(
            mesh = mesh_counter + 1,
            name = node.name
        )
    ])
    print("created new nodes")

    # Link nodes to scene
    gltf_test.scenes[0].nodes.extend([node_counter])

    # Increment all our counters
    node_counter += 2
    mesh_counter += 2
    accessor_counter += 6
    bufferView_counter += 6
    currByteOffset += len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob) + len(triangles_dm_binary_blob)
    
    print("increment successful")
    print("run successful")

working on 0
does not exist need to add
matrices acquired
decimation done
converted to bytes
appended buffer
created new bufferviews
created new accessors
created new meshes
created new nodes
increment successful
run successful


In [148]:
# gltf_test.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
# gltf_test.set_binary_blob(main_binary_blob)

# export_filename = "mep-gulley-trap-test.glb"
# gltf_test.save(export_filename)

In [149]:
binary_blob = gltf_gully.binary_blob()
    
triangles_accessor = gltf_gully.accessors[gltf_gully.meshes[0].primitives[0].indices]
triangles_buffer_view = gltf_gully.bufferViews[triangles_accessor.bufferView]
triangles_array = np.frombuffer(
    binary_blob[
        triangles_buffer_view.byteOffset
        + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
        + triangles_buffer_view.byteLength
    ],
    dtype=GLTF_COMPONENT_TYPES[triangles_accessor.componentType],
    count=triangles_accessor.count,
).reshape((-1, 3))

triangles_array

array([[     2,      9,     17],
       [    21,     26,     37],
       [    40,     45,     57],
       ...,
       [102213, 102202, 102199],
       [102251, 102253, 102244],
       [102254, 102241, 102237]], shape=(38841, 3), dtype=uint32)

In [150]:
points_accessor = gltf_gully.accessors[gltf_gully.meshes[0].primitives[0].attributes.POSITION]
points_buffer_view = gltf_gully.bufferViews[points_accessor.bufferView]
points_array = np.frombuffer(
    binary_blob[
        points_buffer_view.byteOffset
        + points_accessor.byteOffset : points_buffer_view.byteOffset
        + points_buffer_view.byteLength
    ],
    dtype=GLTF_COMPONENT_TYPES[points_accessor.componentType],
    count=points_accessor.count * 3,
).reshape((-1, 3))

points_array

array([[-0.0315548 , -0.29303598,  0.02671323],
       [-0.0315548 , -0.29303598,  0.02671323],
       [-0.0315548 , -0.29303598,  0.02671323],
       ...,
       [ 0.23906961, -0.28738472, -0.035     ],
       [ 0.23906961, -0.28738472, -0.035     ],
       [ 0.23906961, -0.28738472, -0.035     ]],
      shape=(102265, 3), dtype=float32)

We get the same results, but looking at the mesh its not apparent why we have so many vertices. It looks like these are also being repeated. If we export the mesh to wavefront format (.obj), we can open it up in notepad and inspect. We see here that there are 20978 vertices in total, implying that we are definitely counting our vertices more than once.

In [151]:
points_accessor = gltf_gully.accessors[gltf_gully.meshes[0].primitives[0].attributes.POSITION]
points_accessor

Accessor(extensions={}, extras={}, bufferView=0, byteOffset=0, componentType=5126, normalized=False, count=102265, type='VEC3', sparse=None, max=[0.2520062327384949, 0.019497673958539963, 0.2500000298023224], min=[-0.24987658858299255, -0.32313597202301025, -0.25], name=None)

It does seem like we have an extremely high level of precision on this mesh. I count 16 digits after the decimal point. This is obviously too much, and might be why we have such few vertices in our obj file? Lets see if we only keep the unique values of the vertices in our points array above, what will we get?

In [152]:
points_accessor_test = np.unique(points_array, axis=0)
points_accessor_test

array([[-2.4987659e-01, -2.0700000e-01, -2.5000000e-01],
       [-2.4987659e-01, -2.0700000e-01,  2.5000003e-01],
       [-2.4987659e-01, -2.0550001e-01, -2.5000000e-01],
       ...,
       [ 2.5194639e-01, -2.5221616e-01, -6.4845006e-03],
       [ 2.5194642e-01, -2.5221536e-01,  6.4847064e-03],
       [ 2.5200623e-01, -2.5107425e-01,  7.9999998e-09]],
      shape=(20828, 3), dtype=float32)

This might be the problem here, we have too much precision. If we look at the IFC file that generated this shape, we see that it was created using parametrics- i.e. an equation is used to generate the lines. Converting this points will almost definitely explode out the file size.

Let's also get the return index to also do the same to the index array.

In [153]:
points_array_test, unique_indices, unique_inverse = np.unique(points_array, axis=0, return_index=True, return_inverse=True)
unique_inverse

array([ 5167,  5167,  5167, ..., 20633, 20633, 20633], shape=(102265,))

In [154]:
triangles_array_test = unique_inverse[triangles_array]
triangles_array_test

array([[ 5167,  4774,  5671],
       [ 4743,  4414,  5495],
       [12501, 12754, 11155],
       ...,
       [20640, 20638, 20795],
       [20637, 20639, 20804],
       [20639, 20641, 20810]], shape=(38841, 3))

In [155]:
triangles_array_test_unique = np.unique(triangles_array_test, axis=0)
triangles_array_test_unique


array([[    0,  5552,  8338],
       [    0,  8338, 11684],
       [    1,     0,     2],
       ...,
       [20825, 20717, 20751],
       [20826, 20827, 20751],
       [20827, 20823, 20825]], shape=(38841, 3))

In [156]:
normals_accessor = gltf_gully.accessors[gltf_gully.meshes[mesh].primitives[0].attributes.NORMAL]
normals_buffer_view = gltf_gully.bufferViews[normals_accessor.bufferView]
normals_array = np.frombuffer(
    binary_blob[
        normals_buffer_view.byteOffset
        + normals_accessor.byteOffset : normals_buffer_view.byteOffset
        + normals_buffer_view.byteLength
    ],
    dtype=GLTF_COMPONENT_TYPES[normals_accessor.componentType],
    count=normals_accessor.count * 3,
).reshape((-1, 3))

normals_array

array([[ 0.5281981 ,  0.6561977 , -0.53889817],
       [ 0.590881  ,  0.46498504, -0.6592788 ],
       [ 0.6100047 ,  0.4519035 , -0.6509051 ],
       ...,
       [ 0.00680035,  0.13030663, -0.9914504 ],
       [-0.00680035, -0.13030663, -0.9914504 ],
       [-0.9986313 ,  0.05230164,  0.        ]],
      shape=(102265, 3), dtype=float32)

In [157]:
unique_indices

array([86844, 86847, 86802, ...,  5911,  5474,  5478], shape=(20828,))

In [158]:
normals_array_test = normals_array[unique_indices]
normals_array_test

array([[ 0.        ,  0.        ,  1.        ],
       [ 0.        ,  0.        , -1.        ],
       [ 0.        ,  0.        ,  1.        ],
       ...,
       [ 0.13710442,  0.9753315 , -0.17300557],
       [ 0.13709342,  0.97535324,  0.17289172],
       [ 0.13709342,  0.97535324,  0.17289172]],
      shape=(20828, 3), dtype=float32)

Seems like we should be able to reduce the density of the mesh this way. Let's give it a shot.

In [159]:
# Initialize empty GLTF container to store the newly created mesh objects.
gltf_test = pygltflib.GLTF2()
gltf_test.scenes.append(pygltflib.Scene(nodes=[]))
gltf_test.scene = 0

# Initialize Counters and Binary Blob
node_counter = 0
mesh_counter = 0
accessor_counter = 0
bufferView_counter = 0
currByteOffset = 0

main_binary_blob = bytearray()
binary_blob = gltf_gully.binary_blob()

visited = {}

for node in gltf_gully.nodes:

    mesh = node.mesh
    print(f"working on {mesh}")

    if mesh in visited:
        # If mesh exists, use the existing mesh ID. Do not need to create new bufferviews etc.
        ## Decimated comes SECOND
        print("exists")
        gltf_test.nodes.extend([
            pygltflib.Node(
                mesh = visited[mesh],
                rotation = node.rotation,
                translation = node.translation,
                matrix = node.matrix,
                scale = node.scale,
                name = node.name,
                children = [ node_counter + 1 ]
            ),
            pygltflib.Node(
                mesh = visited[mesh] + 1,
                name = node.name
            )
        ])
        gltf_test.scenes[0].nodes.extend([node_counter])
        node_counter += 2

        continue

    # If mesh does not exist yet, first add to dictionary
    print("does not exist need to add")
    visited[mesh] = mesh_counter

    # Get the original index, position and normal matrices
    triangles_accessor = gltf_gully.accessors[gltf_gully.meshes[mesh].primitives[0].indices]
    triangles_buffer_view = gltf_gully.bufferViews[triangles_accessor.bufferView]
    f_matrix = np.frombuffer(
        binary_blob[
            triangles_buffer_view.byteOffset
            + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
            + triangles_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[triangles_accessor.componentType],
        count=triangles_accessor.count,
    ).reshape((-1, 3))

    points_accessor = gltf_gully.accessors[gltf_gully.meshes[mesh].primitives[0].attributes.POSITION]
    points_buffer_view = gltf_gully.bufferViews[points_accessor.bufferView]
    v_matrix = np.frombuffer(
        binary_blob[
            points_buffer_view.byteOffset
            + points_accessor.byteOffset : points_buffer_view.byteOffset
            + points_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[points_accessor.componentType],
        count=points_accessor.count * 3,
    ).reshape((-1, 3))

    normals_accessor = gltf_gully.accessors[gltf_gully.meshes[mesh].primitives[0].attributes.NORMAL]
    normals_buffer_view = gltf_gully.bufferViews[normals_accessor.bufferView]
    n_matrix = np.frombuffer(
        binary_blob[
            normals_buffer_view.byteOffset
            + normals_accessor.byteOffset : normals_buffer_view.byteOffset
            + normals_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[normals_accessor.componentType],
        count=normals_accessor.count * 3,
    ).reshape((-1, 3))
    
    print("matrices acquired")

    # -------- Implement the Precision -----------------
    v_org, unq_indices, unq_inverse = np.unique(v_matrix, axis=0, return_index=True, return_inverse=True)
    f_org = unq_inverse[f_matrix]
    n_org = n_matrix[unq_indices]

    # --------------------------------------------------

    # Apply the decimation and remapping
    v_dm, f_dm, n_dm, v_org_rmp, f_org_rmp, n_org_rmp = decimate_mesh_2(v_org, f_org, n_org)
    print("decimation done")

    f_org_rmp = f_org_rmp.astype(np.uint16)
    f_dm = f_dm.astype(np.uint16)

    v_org_rmp = v_org_rmp.astype(np.float32)
    v_dm = v_dm.astype(np.float32)

    n_org_rmp = n_org_rmp.astype(np.float32)
    n_dm = n_dm.astype(np.float32)

    # Convert to Bytes
    triangles_rmp_binary_blob = f_org_rmp.flatten().tobytes()
    points_rmp_binary_blob = v_org_rmp.tobytes()
    normals_rmp_binary_blob = n_org_rmp.tobytes()
    triangles_dm_binary_blob = f_dm.flatten().tobytes()
    points_dm_binary_blob = v_dm.tobytes()
    normals_dm_binary_blob = n_dm.tobytes()
    print("converted to bytes")

    # Append to Buffer
    main_binary_blob.extend(triangles_rmp_binary_blob + points_rmp_binary_blob + normals_rmp_binary_blob + triangles_dm_binary_blob)
    print("appended buffer")

    # Create New BufferViews
    gltf_test.bufferViews.extend([
                
        ## Indices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset,
            byteLength = len(triangles_rmp_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Indices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob),
            byteLength = len(triangles_dm_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset= currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),
    ])
    print("created new bufferviews")

    # Create new Accessors
    gltf_test.accessors.extend([
                
        ## Indices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_org_rmp.size,
            type = pygltflib.SCALAR,
            max = [int(f_org_rmp.max())],
            min = [int(f_org_rmp.min())],
        ),

        ## Vertices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 1,
            componentType = pygltflib.FLOAT,
            count = len(v_org_rmp),
            type = pygltflib.VEC3,
            max = v_org_rmp.max(axis=0).tolist(),
            min = v_org_rmp.min(axis=0).tolist(),
        ),

        ## Normals - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 2,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_org_rmp),
            type = pygltflib.VEC3,
            max = n_org_rmp.max(axis=0).tolist(),
            min = n_org_rmp.min(axis=0).tolist(),
        ),

        ## Indices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 3,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_dm.size,
            type = pygltflib.SCALAR,
            max = [int(f_dm.max())],
            min = [int(f_dm.min())],
        ),

        ## Vertices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 4,
            componentType = pygltflib.FLOAT,
            count = len(v_dm),
            type = pygltflib.VEC3,
            max = v_dm.max(axis=0).tolist(),
            min = v_dm.min(axis=0).tolist(),
        ),

        ## Normals - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 5,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_dm),
            type = pygltflib.VEC3,
            max = n_dm.max(axis=0).tolist(),
            min = n_dm.min(axis=0).tolist(),
        )
    ])
    print("created new accessors")

    # Create new Meshes
    gltf_test.meshes.extend([
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 1, 
                        NORMAL = accessor_counter + 2,
                    ),
                    indices = accessor_counter + 0
                )
            ]
        ),
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 4, 
                        NORMAL = accessor_counter + 5,
                    ),
                    indices = accessor_counter + 3
                )
            ]
        )
    ])
    print("created new meshes")

    # Create new Nodes
    gltf_test.nodes.extend([
        pygltflib.Node(
            mesh = mesh_counter,
            rotation = node.rotation,
            translation = node.translation,
            scale = node.scale,
            matrix = node.matrix,
            name = node.name,
            children = [ node_counter + 1]
        ),
        pygltflib.Node(
            mesh = mesh_counter + 1,
            name = node.name
        )
    ])
    print("created new nodes")

    # Link nodes to scene
    gltf_test.scenes[0].nodes.extend([node_counter])

    # Increment all our counters
    node_counter += 2
    mesh_counter += 2
    accessor_counter += 6
    bufferView_counter += 6
    currByteOffset += len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob) + len(triangles_dm_binary_blob)
    
    print("increment successful")
    print("run successful")

working on 0
does not exist need to add
matrices acquired
decimation done
converted to bytes
appended buffer
created new bufferviews
created new accessors
created new meshes
created new nodes
increment successful
run successful


In [160]:
# gltf_test.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
# gltf_test.set_binary_blob(main_binary_blob)

# export_filename = "mep-gulley-trap-test-2.glb"
# gltf_test.save(export_filename)

![Gulley Trap Test with Precision Accounted For](../img/gulley-trap-test-with-precision-accounted.png)

That looks much better. Lets try extending this out to our scene once again.

In [ ]:
# Initialize empty GLTF container to store the newly created mesh objects.
gltf_export = pygltflib.GLTF2()
gltf_export.scenes.append(pygltflib.Scene(nodes=[]))
gltf_export.scene = 0

# Initialize Counters and Binary Blob
node_counter = 0
mesh_counter = 0
accessor_counter = 0
bufferView_counter = 0
currByteOffset = 0

main_binary_blob = bytearray()
binary_blob = gltf.binary_blob()

visited = {}

for node in gltf.nodes:

    mesh = node.mesh
    print(f"working on {mesh}")

    if mesh in visited:
        # If mesh exists, use the existing mesh ID. Do not need to create new bufferviews etc.
        ## Decimated comes SECOND
        print("exists")
        gltf_export.nodes.extend([
            pygltflib.Node(
                mesh = visited[mesh],
                rotation = node.rotation,
                translation = node.translation,
                matrix = node.matrix,
                scale = node.scale,
                name = node.name,
                children = [ node_counter + 1 ]
            ),
            pygltflib.Node(
                mesh = visited[mesh] + 1,
                name = node.name
            )
        ])
        gltf_export.scenes[0].nodes.extend([node_counter])
        node_counter += 2

        continue

    # If mesh does not exist yet, first add to dictionary
    print("does not exist need to add")
    visited[mesh] = mesh_counter

    # Get the original index, position and normal matrices
    triangles_accessor = gltf.accessors[gltf.meshes[mesh].primitives[0].indices]
    triangles_buffer_view = gltf.bufferViews[triangles_accessor.bufferView]
    f_matrix = np.frombuffer(
        binary_blob[
            triangles_buffer_view.byteOffset
            + triangles_accessor.byteOffset : triangles_buffer_view.byteOffset
            + triangles_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[triangles_accessor.componentType],
        count=triangles_accessor.count,
    ).reshape((-1, 3))

    points_accessor = gltf.accessors[gltf.meshes[mesh].primitives[0].attributes.POSITION]
    points_buffer_view = gltf.bufferViews[points_accessor.bufferView]
    v_matrix = np.frombuffer(
        binary_blob[
            points_buffer_view.byteOffset
            + points_accessor.byteOffset : points_buffer_view.byteOffset
            + points_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[points_accessor.componentType],
        count=points_accessor.count * 3,
    ).reshape((-1, 3))

    normals_accessor = gltf.accessors[gltf.meshes[mesh].primitives[0].attributes.NORMAL]
    normals_buffer_view = gltf.bufferViews[normals_accessor.bufferView]
    n_matrix = np.frombuffer(
        binary_blob[
            normals_buffer_view.byteOffset
            + normals_accessor.byteOffset : normals_buffer_view.byteOffset
            + normals_buffer_view.byteLength
        ],
        dtype=GLTF_COMPONENT_TYPES[normals_accessor.componentType],
        count=normals_accessor.count * 3,
    ).reshape((-1, 3))
    
    print("matrices acquired")

    # -------- Implement the Precision -----------------
    v_org, unq_indices, unq_inverse = np.unique(v_matrix, axis=0, return_index=True, return_inverse=True)
    f_org = unq_inverse[f_matrix]
    n_org = n_matrix[unq_indices]

    # --------------------------------------------------

    # Apply the decimation and remapping
    v_dm, f_dm, n_dm, v_org_rmp, f_org_rmp, n_org_rmp = decimate_mesh_2(v_org, f_org, n_org, 0.2)
    print("decimation done")

    f_org_rmp = f_org_rmp.astype(np.uint16)
    f_dm = f_dm.astype(np.uint16)

    v_org_rmp = v_org_rmp.astype(np.float32)
    v_dm = v_dm.astype(np.float32)

    n_org_rmp = n_org_rmp.astype(np.float32)
    n_dm = n_dm.astype(np.float32)

    # Convert to Bytes
    triangles_rmp_binary_blob = f_org_rmp.flatten().tobytes()
    points_rmp_binary_blob = v_org_rmp.tobytes()
    normals_rmp_binary_blob = n_org_rmp.tobytes()
    triangles_dm_binary_blob = f_dm.flatten().tobytes()
    points_dm_binary_blob = v_dm.tobytes()
    normals_dm_binary_blob = n_dm.tobytes()
    print("converted to bytes")

    # Append to Buffer
    main_binary_blob.extend(triangles_rmp_binary_blob + points_rmp_binary_blob + normals_rmp_binary_blob + triangles_dm_binary_blob)
    print("appended buffer")

    # Create New BufferViews
    gltf_export.bufferViews.extend([
                
        ## Indices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset,
            byteLength = len(triangles_rmp_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Remapped
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_rmp_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Indices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob),
            byteLength = len(triangles_dm_binary_blob),
            target=pygltflib.ELEMENT_ARRAY_BUFFER
        ),

        ## Vertices - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset= currByteOffset + len(triangles_rmp_binary_blob),
            byteLength = len(points_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),

        ## Normals - Decimated
        pygltflib.BufferView(
            buffer = 0,
            byteOffset = currByteOffset + len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob),
            byteLength = len(normals_dm_binary_blob),
            target=pygltflib.ARRAY_BUFFER
        ),
    ])
    print("created new bufferviews")

    # Create new Accessors
    gltf_export.accessors.extend([
                
        ## Indices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_org_rmp.size,
            type = pygltflib.SCALAR,
            max = [int(f_org_rmp.max())],
            min = [int(f_org_rmp.min())],
        ),

        ## Vertices - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 1,
            componentType = pygltflib.FLOAT,
            count = len(v_org_rmp),
            type = pygltflib.VEC3,
            max = v_org_rmp.max(axis=0).tolist(),
            min = v_org_rmp.min(axis=0).tolist(),
        ),

        ## Normals - Original Remapped
        pygltflib.Accessor(
            bufferView = bufferView_counter + 2,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_org_rmp),
            type = pygltflib.VEC3,
            max = n_org_rmp.max(axis=0).tolist(),
            min = n_org_rmp.min(axis=0).tolist(),
        ),

        ## Indices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 3,
            componentType = pygltflib.UNSIGNED_SHORT,
            count = f_dm.size,
            type = pygltflib.SCALAR,
            max = [int(f_dm.max())],
            min = [int(f_dm.min())],
        ),

        ## Vertices - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 4,
            componentType = pygltflib.FLOAT,
            count = len(v_dm),
            type = pygltflib.VEC3,
            max = v_dm.max(axis=0).tolist(),
            min = v_dm.min(axis=0).tolist(),
        ),

        ## Normals - Decimated
        pygltflib.Accessor(
            bufferView = bufferView_counter + 5,
            byteOffset = 0,
            componentType = pygltflib.FLOAT,
            count = len(n_dm),
            type = pygltflib.VEC3,
            max = n_dm.max(axis=0).tolist(),
            min = n_dm.min(axis=0).tolist(),
        )
    ])
    print("created new accessors")

    # Create new Meshes
    gltf_export.meshes.extend([
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 1, 
                        NORMAL = accessor_counter + 2,
                    ),
                    indices = accessor_counter + 0
                )
            ]
        ),
        pygltflib.Mesh(
            primitives=[
                pygltflib.Primitive(
                    attributes=pygltflib.Attributes(
                        POSITION = accessor_counter + 4, 
                        NORMAL = accessor_counter + 5,
                    ),
                    indices = accessor_counter + 3
                )
            ]
        )
    ])
    print("created new meshes")

    # Create new Nodes
    gltf_export.nodes.extend([
        pygltflib.Node(
            mesh = mesh_counter,
            rotation = node.rotation,
            translation = node.translation,
            scale = node.scale,
            matrix = node.matrix,
            name = node.name,
            children = [ node_counter + 1]
        ),
        pygltflib.Node(
            mesh = mesh_counter + 1,
            name = node.name
        )
    ])
    print("created new nodes")

    # Link nodes to scene
    gltf_export.scenes[0].nodes.extend([node_counter])

    # Increment all our counters
    node_counter += 2
    mesh_counter += 2
    accessor_counter += 6
    bufferView_counter += 6
    currByteOffset += len(triangles_rmp_binary_blob) + len(points_rmp_binary_blob) + len(normals_rmp_binary_blob) + len(triangles_dm_binary_blob)
    
    print("increment successful")
    print("run successful")

working on 0
does not exist need to add
matrices acquired
decimation done
converted to bytes
appended buffer
created new bufferviews
created new accessors
created new meshes
created new nodes
increment successful
run successful
working on 0
exists
working on 1
does not exist need to add
matrices acquired
decimation done
converted to bytes
appended buffer
created new bufferviews
created new accessors
created new meshes
created new nodes
increment successful
run successful
working on 1
exists
working on 1
exists
working on 1
exists
working on 1
exists
working on 1
exists
working on 1
exists
working on 2
does not exist need to add
matrices acquired
decimation done
converted to bytes
appended buffer
created new bufferviews
created new accessors
created new meshes
created new nodes
increment successful
run successful
working on 2
exists
working on 2
exists
working on 2
exists
working on 2
exists
working on 2
exists
working on 2
exists
working on 3
does not exist need to add
matrices acquire

In [162]:
for scene in gltf_export.scenes:
    scene.extras = {
        "nInstances": 23234,
        "vCount": 5494367,
        "iCount": 35104182
    }
    print(scene.extras)

{'nInstances': 23234, 'vCount': 5494367, 'iCount': 35104182}


In [163]:
gltf_export.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
gltf_export.set_binary_blob(main_binary_blob)

export_filename = "mep-fittings_hi-lo-res_ifc-guid-2.glb"
gltf_export.save(export_filename)

True

And finally, it looks like we have a good export.

Rather interestingly, it seems like the precision modelling helped the most here since our file size has actually reduced by quite a fair bit- 18 MB versus 32 MB.

Let's see what this model looks like in the scene.

The last step now will be to append the prebatching information to our scene level. We have precomputed this elsewhere and so will be hardcoding this information, however it is pretty easy to manually acquire if needed.

In [84]:
filename = "mep-fittings_hi-lo-res_ifc-guid-2.glb"
gltf = pygltflib.GLTF2().load(filename)

In [85]:
for scene in gltf.scenes:
    scene.extras = {
        "nInstances": 23234,
        "vCount": 5494367,
        "iCount": 35104182
    }
    print(scene.extras)

{'nInstances': 23234, 'vCount': 5494367, 'iCount': 35104182}


In [ ]:
# gltf_export.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
# gltf_export.set_binary_blob(main_binary_blob)

# export_filename = "mep-fittings_hi-lo-res_ifc-guid-2.glb"
# gltf.save(export_filename)

True